# Baseline: TF-IDF + Logistic Regression

По истории действий клиента предсказываем `target_product` — код продукта следующего целевого клика. Метрика — Accuracy (HitRate@1).

В EDA целевой продукт встречался в 90,53% историй, а классы оказались несбалансированы. Начнём с частотного представления действий и продуктов: TF-IDF преобразует историю в разреженный вектор, а Logistic Regression выбирает один из классов. Сравним результат с предсказанием самого частого продукта.

Это первый этап: только train, предобработка и одна валидация. Ячейки подготовки данных и обучения нужно выполнить вручную; результаты заранее не заданы.

## 1. Загрузка данных и разделение выборки

Загрузим правильные ответы (`target_product`), идентификаторы клиентов (`client_id`) и справочник событий.

Разделим данные на обучение и валидацию в пропорции примерно 80/20. Используем `GroupShuffleSplit`, чтобы истории одного клиента не попадали в обе выборки. Это позволит проверить качество модели на новых клиентах.

In [8]:
from pathlib import Path
from collections import Counter
from itertools import repeat
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

DATA_DIR = Path('data') if Path('data/train.parquet').exists() else Path('task3/data')
BATCH_SIZE = 32
RANDOM_STATE = 42

metadata = pq.read_table(DATA_DIR / 'train.parquet', columns=['client_id', 'target_product']).to_pandas() # пока только client_id и target
event_dictionary = pq.read_table(DATA_DIR / 'event_dictionary.parquet').to_pandas()                        
event_types = event_dictionary.set_index('encoded_event')['event_type'].to_dict()                         # словарь "код: событие"

y = metadata['target_product'].to_numpy()
groups = metadata['client_id'].to_numpy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)  
train_idx, valid_idx = next(splitter.split(metadata, y, groups=groups))             # делим по клиентам (не по историям)
y_train, y_valid = y[train_idx], y[valid_idx]
assert set(groups[train_idx]).isdisjoint(groups[valid_idx])

print(f'Обучение: {len(train_idx):,} историй; валидация: {len(valid_idx):,}')
print('Классов в обучении, валидации:', len(np.unique(y_train)), ',', len(np.unique(y_valid)))
print('Классы валидации, отсутствующие в обучении:', sorted(set(y_valid) - set(y_train)))

Обучение: 80,000 историй; валидация: 20,000
Классов в обучении, валидации: 32 , 31
Классы валидации, отсутствующие в обучении: []


## 2. Подготовка историй клиентов

Для каждой истории посчитаем, какие действия совершал клиент и с какими продуктами взаимодействовал. Для подсчёта используем `Counter`. Истории будем читать небольшими частями, чтобы не перегружать память.

Полученные данные затем преобразуем в числовые признаки с помощью TF-IDF.

In [9]:
# считаем действия и продукты в одной истории
def history_tokens(events):
    tokens = Counter()
    for event in events or []:
        kind = event_types.get(event['encoded_event'], 'unknown') if event else 'unknown'
        product = event['encoded_product'] if event else None

        tokens[f'a_{kind}'] += 1
        if product is None:
            tokens[f'a_{kind}_p_missing'] += 1
        else:
            tokens[f'p_{product}'] += 1
            tokens[f'a_{kind}_p_{product}'] += 1

    return tokens or Counter({'empty_history': 1})

# словарь "токен-количество" -> повторы токенов 
def counted_tokens(document):
    for token, count in document.items():
        yield from repeat(token, count)


# читаем данные по частям и считаем события
documents = []
for batch in pq.ParquetFile(DATA_DIR / 'train.parquet').iter_batches(
        batch_size=BATCH_SIZE, columns=['events']):
    for history in batch.column('events').to_pylist():
        documents.append(history_tokens(history))

del batch, history
assert len(documents) == len(metadata)

train_documents = [documents[i] for i in train_idx]
valid_documents = [documents[i] for i in valid_idx]

del documents

## 3. Преобразование историй с помощью TF-IDF

С помощью TF-IDF превратим частоты действий и продуктов в числовые признаки.
Сначала вычислим TF-IDF на обучающих историях (`fit_transform`), а затем по тем же правилам преобразуем валидационные истории (`transform`).
В результате получим две таблицы числовых признаков, которые передадим логистической регрессии.

In [ ]:
vectorizer = TfidfVectorizer(
    analyzer=counted_tokens,    # токены с учётом кол-ва повторений
    min_df=2,                   # токен встречается хотя бы в 2 историях
    max_features=20_000,        # не больше 20000 разных токенов
    dtype=np.float32,
)

X_train = vectorizer.fit_transform(train_documents) # считаем tf-idf 
del train_documents
X_valid = vectorizer.transform(valid_documents)     # считаем tf и умножаем на уже известный idf
del valid_documents

print('Размеры матриц:', X_train.shape, '/', X_valid.shape)
print('Число токенов в словаре:', len(vectorizer.vocabulary_))

Размеры матриц: (80000, 231) / (20000, 231)
Число токенов в словаре: 231


## 4. Обучение и проверка модели

Обучим логистическую регрессию на TF-IDF-признаках и проверим её Accuracy на валидации.

Для сравнения используем простую стратегию: всегда предсказывать самый популярный продукт из обучающей выборки. Так мы сможем оценить, насколько наша модель лучше обычного угадывания по частотам.

In [ ]:
model = LogisticRegression(
    solver='saga',             
    C=1.0,                      
    max_iter=300, 
    tol=1e-3, 
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)
valid_prediction = model.predict(X_valid)

majority_product = pd.Series(y_train).value_counts().idxmax() # самый популярный продукт
majority_prediction = np.full(len(y_valid), majority_product)

majority_accuracy = accuracy_score(y_valid, majority_prediction)
model_accuracy = accuracy_score(y_valid, valid_prediction)

display(pd.DataFrame({
    'Стратегия': ['Самый частый продукт', 'TF-IDF + Logistic Regression'],
    'Accuracy': [majority_accuracy, model_accuracy],
}).round(4))

print(f'Разница Accuracy: {(model_accuracy - majority_accuracy) * 100:+.2f} п.п.')
print('Максимальное число выполненных итераций:', model.n_iter_.max())

,Стратегия,Accuracy
0,Самый частый продукт,0.1573
1,TF-IDF + Logistic Regression,0.3512


Разница Accuracy: +19.39 п.п.
Максимальное число выполненных итераций: 15


## Выводы

Мы построили первую модель для предсказания следующего продукта клиента. Истории действий преобразовали в числовые признаки с помощью TF-IDF, после чего обучили логистическую регрессию.

На валидации получили следующие результаты:

- Предсказание самого популярного продукта — **Accuracy 0.1573**.
- TF-IDF + Logistic Regression — **Accuracy 0.3512**.

Наша модель правильно предсказывает продукт примерно в 35% случаев и заметно превосходит простейшую стратегию. Это показывает, что предыдущие действия клиента содержат полезную информацию для прогноза.

При этом baseline учитывает только частоты действий и продуктов, но не их порядок и время. В дальнейшем попробуем добавить признаки последних взаимодействий и обучить LightGBM или CatBoost для повышения качества.